# PAR-S V2 Task13 Formal550 Acceptance Review

> **Authority: `informational_read_only`.** This notebook explains existing,
> SHA-bound automatic evidence. It **does not define, override, or write PASS/FAIL**,
> does not collect human notes, and cannot authorize another
> generation or change any frozen dataset artifact.

Scope: the immutable 500-case main role and independent 50-case negative role.
The authoritative result remains the Task4 automatic acceptance JSON supplied
to this notebook.


## 1. Frozen inputs and authority boundary

The setup reads the automatic acceptance JSON, verifies each referenced gate
against the SHA-256 recorded in `gate_rows`, and reads both role manifests from
their immutable roots. Projection paths are assembled into a display-only
`visual_registry`. Each selected projection is read once, checked against the
manifest size and SHA-256, and displayed from that verified in-memory byte
snapshot; files are never opened in a writable mode.


In [ ]:
from pathlib import Path
import hashlib
import json

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import JSON as DisplayJSON, Markdown, display

ACCEPTANCE_JSON = Path("D:\\PFE-U\\PAR\\outputs\\pars_v2_formal550_v1_qa\\TASK13_FORMAL550_AUTOMATIC_ACCEPTANCE.json")
MAIN_ROOT = Path("D:\\PFE-U\\PAR\\outputs\\pars_v2_formal550_v1\\main")
NEGATIVE_ROOT = Path("D:\\PFE-U\\PAR\\outputs\\pars_v2_formal550_v1\\negative")
EXPECTED_ACCEPTANCE_SCHEMA = "pars_v2_task13_formal550_automatic_acceptance_v1"
EXPECTED_GENERATOR_SCHEMA = "formal550_generator_gate_v1"
EXPECTED_PROJECTION_SHAPE = (60, 128, 128)


def read_json_bytes(path):
    payload = Path(path).read_bytes()
    value = json.loads(payload)
    if not isinstance(value, dict):
        raise TypeError(f"{path} must contain a JSON object")
    return value, payload


def read_bound_gate(row):
    value, payload = read_json_bytes(Path(row["path"]))
    if hashlib.sha256(payload).hexdigest() != row["sha256"]:
        raise ValueError(f"SHA-256 mismatch for {row['gate_id']}")
    if value.get("status") != row["status"]:
        raise ValueError(f"status mismatch for {row['gate_id']}")
    return value


def load_role_manifest(role, root, expected_manifest_sha256):
    role_root = root.resolve()
    marker, _ = read_json_bytes(root / "DATASET_COMPLETE.json")
    if marker.get("status") != "complete" or marker.get("dataset_role") != role:
        raise ValueError(f"{role} completion marker identity/status mismatch")
    manifest_relative = Path(marker["manifest_relative_path"])
    if manifest_relative.is_absolute():
        raise ValueError(f"{role} manifest path must be relative")
    manifest_path = (role_root / manifest_relative).resolve()
    try:
        manifest_path.relative_to(role_root)
    except ValueError as exc:
        raise ValueError(f"{role} manifest path escapes role root") from exc
    manifest_payload = manifest_path.read_bytes()
    actual_manifest_sha256 = hashlib.sha256(manifest_payload).hexdigest()
    if actual_manifest_sha256 != expected_manifest_sha256:
        raise ValueError(f"{role} generator gate manifest SHA-256 mismatch")
    if actual_manifest_sha256 != marker["manifest_sha256"]:
        raise ValueError(f"{role} manifest SHA-256 mismatch")
    rows = [json.loads(line) for line in manifest_payload.splitlines() if line.strip()]
    if len(rows) != marker["case_count"]:
        raise ValueError(f"{role} manifest count mismatch")
    return marker, rows


def resolve_role_artifact(root, artifact):
    role_root = root.resolve()
    relative = Path(artifact["relative_path"])
    if relative.is_absolute():
        raise ValueError("projection artifact path must be relative")
    path = (role_root / relative).resolve()
    try:
        path.relative_to(role_root)
    except ValueError as exc:
        raise ValueError("projection artifact path escapes role root") from exc
    return {
        "path": path,
        "size_bytes": int(artifact["size_bytes"]),
        "sha256": str(artifact["sha256"]),
    }


automatic, _ = read_json_bytes(ACCEPTANCE_JSON)
assert automatic["schema_version"] == EXPECTED_ACCEPTANCE_SCHEMA
assert automatic["notebook_authority"] == "informational_read_only"

gate_documents = {
    row["gate_id"]: read_bound_gate(row) for row in automatic['gate_rows']
}
generator_gate = gate_documents["formal550_generator_gate_v1"]
main_loader_gate = gate_documents["formal550_main_loader_gate_v1"]
negative_loader_gate = gate_documents["formal550_negative_loader_gate_v1"]
coordinate_gate = gate_documents["projection_coordinate_gate_v2"]
assert generator_gate["schema_version"] == EXPECTED_GENERATOR_SCHEMA

main_marker, main_manifest_rows = load_role_manifest(
    "main", MAIN_ROOT, generator_gate["dataset_manifests"]["main"]
)
negative_marker, negative_manifest_rows = load_role_manifest(
    "negative",
    NEGATIVE_ROOT,
    generator_gate["dataset_manifests"]["negative"],
)
assert main_marker["case_count"] == automatic["role_case_counts"]["main"]
assert negative_marker["case_count"] == automatic["role_case_counts"]["negative"]

role_roots = {"main": MAIN_ROOT, "negative": NEGATIVE_ROOT}
visual_registry = {}
for role, rows in (
    ("main", main_manifest_rows),
    ("negative", negative_manifest_rows),
):
    visual_registry[role] = {
        row["case_id"]: resolve_role_artifact(
            role_roots[role], row["artifacts"]["projection_a00"]
        )
        for row in rows
    }

case_frame = pd.DataFrame(generator_gate["cases"])
assert len(case_frame) == automatic["case_count"]
for role in ("main", "negative"):
    audited = set(case_frame.loc[case_frame["dataset_role"] == role, "case_id"])
    if audited != set(visual_registry[role]):
        raise ValueError(f"{role} generator/manifest case set mismatch")

display(pd.DataFrame([{
    "authority": automatic["notebook_authority"],
    "automatic_status": automatic["status"],
    "automatic_gate_passed": automatic["automatic_gate_passed"],
    "case_count": automatic["case_count"],
    "main_cases": automatic["role_case_counts"]["main"],
    "negative_cases": automatic["role_case_counts"]["negative"],
}]))


## 2. Authoritative gate structure

```mermaid
flowchart LR
    A["Formal550 generator artifact/statistical gate"] --> E["Task4 automatic acceptance JSON"]
    B["Main PAR-S_2 loader gate"] --> E
    C["Negative PAR-S_2 loader gate"] --> E
    D["Frozen projection coordinate gate"] --> E
    E --> F["This informational read-only review"]
```

The notebook displays the statuses already recorded by Task4. It does not
derive a replacement decision from the tables or plots below.


In [ ]:
gate_frame = pd.DataFrame(automatic['gate_rows'])[
    ["gate_id", "blocking", "status", "schema_version", "path", "sha256"]
]
display(gate_frame)
display(DisplayJSON({
    "coordinate_contract": coordinate_gate["projection_coordinates"],
    "main_loader_status": main_loader_gate["status"],
    "negative_loader_status": negative_loader_gate["status"],
}))


## 3. Main and negative role/split summaries

The main role contains the tumor-bearing population cohort; the independent
negative role is a test-only, zero-tumor control cohort. Counts below come from
the frozen generator and loader reports, not from notebook-authored criteria.


In [ ]:
role_summary = (
    case_frame.groupby(["dataset_role", "status"], sort=True)
    .size()
    .rename("case_count")
    .reset_index()
)
split_summary = (
    case_frame.groupby(["dataset_role", "split"], sort=True)
    .size()
    .rename("case_count")
    .reset_index()
)
loader_summary = pd.DataFrame([
    {
        "dataset_role": role,
        "status": gate["status"],
        "expected_count": gate.get("expected_count"),
        "observed_count": gate.get("observed_count"),
    }
    for role, gate in (
        ("main", main_loader_gate),
        ("negative", negative_loader_gate),
    )
])
display(role_summary)
display(split_summary)
display(loader_summary)


## 4. Cohort distributions

These plots expose the measured projection distributions separately for main
and negative roles. They are descriptive views of the generator gate's frozen
per-case rows; no plotted value creates or changes a threshold.


In [ ]:
distribution_fields = [
    "projection_weight_sum",
    "view_sum_cv",
    "view_sum_ratio",
    "minimum_positive_bin_fraction_per_view",
    "outer_8px_count_fraction",
]
fig, axes = plt.subplots(
    len(distribution_fields), 1, figsize=(10, 3.0 * len(distribution_fields))
)
for axis, field in zip(axes, distribution_fields):
    for role, color in (("main", "#285f8f"), ("negative", "#b4513e")):
        values = case_frame.loc[case_frame["dataset_role"] == role, field].astype(float)
        axis.hist(values, bins=min(20, max(1, len(values))), alpha=0.55, label=role, color=color)
    axis.set_title(field.replace("_", " "))
    axis.set_ylabel("case count")
    axis.legend()
axes[-1].set_xlabel("reported value")
fig.tight_layout()
plt.show()


## 5. Projection metrics

The aggregate table reproduces the min/median/mean/max values from the formal
generator gate. The per-case table keeps role and split visible so extrema and
focus-case reasons remain auditable.


In [ ]:
projection_summary_rows = []
for role, summaries in generator_gate["projection_statistics"].items():
    for metric, summary in summaries.items():
        projection_summary_rows.append({
            "dataset_role": role,
            "metric": metric,
            **summary,
        })
display(pd.DataFrame(projection_summary_rows))
display(case_frame[[
    "case_id",
    "dataset_role",
    "split",
    "status",
    *distribution_fields,
]])


## 6. Main and negative focus-case projection sliders

Task4 selected focus cases deterministically from automatic attention cases and
per-role projection extrema. Each role has its own case selector and view
slider. Stored SIMIND view `v` is labelled in all frozen angle bases:

- `SIMIND = (180° + 6°v) mod 360°`, clockwise-positive;
- `projector = (90° + 6°v) mod 360°`, clockwise-positive;
- `clinical DICOM camera = (270° - projector) mod 360°`.

The detector-v flip shown here is the frozen loader transform. Controls affect
only the displayed projection; they do not mutate evidence or gate outcomes.


In [ ]:
focus_frame = pd.DataFrame(generator_gate["focus_cases"])
display(focus_frame)


def role_focus_case_ids(role):
    selected = [
        item["case_id"]
        for item in generator_gate["focus_cases"]
        if item["dataset_role"] == role
    ]
    if selected:
        return tuple(selected)
    return tuple(sorted(visual_registry[role]))


def load_projection(role, case_id):
    artifact = visual_registry[role][case_id]
    path = artifact["path"]
    payload = path.read_bytes()
    if len(payload) != artifact["size_bytes"]:
        raise ValueError(f"{case_id} projection size mismatch")
    if hashlib.sha256(payload).hexdigest() != artifact["sha256"]:
        raise ValueError(f"{case_id} projection SHA-256 mismatch")
    expected_bytes = int(np.prod(EXPECTED_PROJECTION_SHAPE)) * np.dtype("<f4").itemsize
    if len(payload) != expected_bytes:
        raise ValueError(f"{case_id} projection byte size mismatch")
    return np.frombuffer(payload, dtype="<f4").reshape(EXPECTED_PROJECTION_SHAPE)


focus_projection_snapshots = {
    (role, case_id): load_projection(role, case_id)
    for role in ("main", "negative")
    for case_id in role_focus_case_ids(role)
}


def show_focus_projection(role, case_id, view):
    projection = focus_projection_snapshots[(role, case_id)]
    canonical_image = np.asarray(projection[view, ::-1, :])
    simind_angle = (180.0 + view * 6.0) % 360.0
    projector_angle = (90.0 + view * 6.0) % 360.0
    clinical_angle = (270.0 - projector_angle) % 360.0
    degree = chr(176)
    fig, axis = plt.subplots(figsize=(6.8, 5.8), constrained_layout=True)
    artist = axis.imshow(np.log1p(canonical_image), cmap="magma", origin="lower")
    axis.set_title(
        f"{role} | {case_id} | canonical view {view:02d}\n"
        f"SIMIND {simind_angle:.1f}{degree} CW+  ↔  "
        f"projector {projector_angle:.1f}{degree} CW+  ↔  "
        f"clinical DICOM camera {clinical_angle:.1f}{degree}"
    )
    axis.set_xlabel("detector u")
    axis.set_ylabel("detector v (frozen loader flip applied)")
    fig.colorbar(artist, ax=axis, label="log1p SIMIND weight")
    plt.show()


main_focus_case_slider = widgets.SelectionSlider(
    options=role_focus_case_ids("main"),
    description="Main focus case",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="760px"),
)
main_projection_view_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=59,
    step=1,
    description="Main projection view",
    continuous_update=False,
    readout_format="02d",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="760px"),
)
negative_focus_case_slider = widgets.SelectionSlider(
    options=role_focus_case_ids("negative"),
    description="Negative focus case",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="760px"),
)
negative_projection_view_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=59,
    step=1,
    description="Negative projection view",
    continuous_update=False,
    readout_format="02d",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="760px"),
)

display(Markdown("### Main-role focus projection"))
display(widgets.interactive(
    lambda case_id, view: show_focus_projection("main", case_id, view),
    case_id=main_focus_case_slider,
    view=main_projection_view_slider,
))
display(Markdown("### Negative-role focus projection"))
display(widgets.interactive(
    lambda case_id, view: show_focus_projection("negative", case_id, view),
    case_id=negative_focus_case_slider,
    view=negative_projection_view_slider,
))


## 7. Read-only conclusion

The final cell repeats the automatic status and evidence boundary verbatim.
Only the Task4 JSON is authoritative; interaction above changes display state
only.


In [ ]:
display(Markdown(
    f"### Automatic status: **{automatic['status'].upper()}**\n\n"
    f"- Automatic gate passed: `{automatic['automatic_gate_passed']}`\n"
    f"- Notebook authority: `{automatic['notebook_authority']}`\n"
    f"- Cases reviewed: `{automatic['case_count']}`"
))
display(pd.DataFrame(automatic['gate_rows'])[["gate_id", "status", "sha256"]])
